- [x] Collect ratings, Compute mean rating + Mean-center ratings 
- [ ] Build user profile
- [ ] Apply time decay weights
- [ ] Recent releases boost

We use content-based filtering by computing similarity between movies.

In [1]:
import pandas as pd
from utils.db import get_connection
from utils.movie import load_movies

con = get_connection()
df = load_movies(con)
df = df.drop('note', axis=1)
df.tail()

,id,name,year,status,type,country,genres,rating,watched_date
353,354,Pearl,2022,waiting,movie,US,"horror,thriller",NaN,None
354,355,High Life,2018,waiting,movie,US,"sci-fi,thriller",NaN,None
355,356,Saltburn,2023,waiting,movie,US,"thriller,dark comedy,psychological,drama",NaN,None
356,357,Bloody Flower,2026,waiting,series,Korea,"mystery,crime,thriller,psychological",NaN,None
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi",NaN,None


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358 entries, 0 to 357
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            358 non-null    int64  
 1   name          358 non-null    object 
 2   year          358 non-null    int64  
 3   status        358 non-null    object 
 4   type          358 non-null    object 
 5   country       346 non-null    object 
 6   genres        358 non-null    object 
 7   rating        70 non-null     float64
 8   watched_date  70 non-null     object 
dtypes: float64(1), int64(2), object(6)
memory usage: 25.3+ KB


## Preprocessing

In [3]:
# watched_df = df[df['status'] == 'completed']

# unwatched_df = df[df['status'] == 'waiting']
# unwatched_df = unwatched_df.drop(columns=['status', 'rating', 'watched_date'])
# unwatched_df.head()

In [4]:
df = df.drop(columns=['rating', 'watched_date'])
df = df.reset_index(drop=True)
df.tail()

,id,name,year,status,type,country,genres
353,354,Pearl,2022,waiting,movie,US,"horror,thriller"
354,355,High Life,2018,waiting,movie,US,"sci-fi,thriller"
355,356,Saltburn,2023,waiting,movie,US,"thriller,dark comedy,psychological,drama"
356,357,Bloody Flower,2026,waiting,series,Korea,"mystery,crime,thriller,psychological"
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi"


In [5]:
df.shape

(358, 7)

### Year

In [6]:
# Min-Max scaling for year, because: below-average year becomes a penalty -> not meaningful
# Min-Max scaling for year does not mean: the more recent the better

df_features = df.copy()
min_year, max_year =  df['year'].min(), df['year'].max()
df_features['year'] = (df['year'] - min_year) / (max_year - min_year)
df_features.tail()

,id,name,year,status,type,country,genres
353,354,Pearl,0.942029,waiting,movie,US,"horror,thriller"
354,355,High Life,0.884058,waiting,movie,US,"sci-fi,thriller"
355,356,Saltburn,0.956522,waiting,movie,US,"thriller,dark comedy,psychological,drama"
356,357,Bloody Flower,1.000000,waiting,series,Korea,"mystery,crime,thriller,psychological"
357,358,Hoppers,1.000000,waiting,movie,US,"comedy,animation,sci-fi"


### Rating

rating is used when building user profile

In [ ]:
# print('Rating mean: ', watched_df['rating'].mean())

In [8]:
# # Mean-centered rating
# df['mean_rating'] = df['rating'] - df['rating'].mean()
# sns.boxplot(df['mean_rating'].dropna())

In [9]:
# liked_df = df[df['rating'] >= 8]
# liked_df.tail(), liked_df.shape

## Feature similarity matrix

In [10]:
df.shape

(358, 7)

In [11]:
df.tail()

,id,name,year,status,type,country,genres
353,354,Pearl,2022,waiting,movie,US,"horror,thriller"
354,355,High Life,2018,waiting,movie,US,"sci-fi,thriller"
355,356,Saltburn,2023,waiting,movie,US,"thriller,dark comedy,psychological,drama"
356,357,Bloody Flower,2026,waiting,series,Korea,"mystery,crime,thriller,psychological"
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi"


### Type

In [12]:
type_features = df['type'].str.get_dummies()
type_features.tail()

,movie,series
353,1,0
354,1,0
355,1,0
356,0,1
357,1,0


### Country

In [13]:
df['country'] = df['country'].fillna('Unknown')
country_features = df['country'].str.get_dummies()
country_features.tail()

,China,Japan,Korea,US,Unknown
353,0,0,0,1,0
354,0,0,0,1,0
355,0,0,0,1,0
356,0,0,1,0,0
357,0,0,0,1,0


### Genres

In [14]:
genres_features = df['genres'].str.get_dummies(sep=',')
genres_features.tail()

,action,adventure,animation,biography,comedy,coming of age,crime,dark comedy,death,documentary,...,sci-fi,sitcom,sport,supernatural,survival,thriller,time travel,war,youth,zombie
353,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
354,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
355,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
356,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
357,0,0,1,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [15]:
# genres_dict = df.groupby('genres').size().to_dict()
# genres = set(genres_dict.keys())
# print(genres)

In [16]:
# df.groupby('genres')['rating'].mean().sort_values(ascending=False)

In [17]:
df_features.tail()

,id,name,year,status,type,country,genres
353,354,Pearl,0.942029,waiting,movie,US,"horror,thriller"
354,355,High Life,0.884058,waiting,movie,US,"sci-fi,thriller"
355,356,Saltburn,0.956522,waiting,movie,US,"thriller,dark comedy,psychological,drama"
356,357,Bloody Flower,1.000000,waiting,series,Korea,"mystery,crime,thriller,psychological"
357,358,Hoppers,1.000000,waiting,movie,US,"comedy,animation,sci-fi"


In [18]:
features = pd.concat(
    [
        df_features.drop(columns=['id', 'name', 'status', 'type', 'country', 'genres']), 
        type_features, country_features, genres_features
    ], axis=1
)
features.tail()

,year,movie,series,China,Japan,Korea,US,Unknown,action,adventure,...,sci-fi,sitcom,sport,supernatural,survival,thriller,time travel,war,youth,zombie
353,0.942029,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
354,0.884058,1,0,0,0,0,1,0,0,0,...,1,0,0,0,0,1,0,0,0,0
355,0.956522,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
356,1.000000,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
357,1.000000,1,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [19]:
features.shape

(358, 48)

In [20]:
df.tail()

,id,name,year,status,type,country,genres
353,354,Pearl,2022,waiting,movie,US,"horror,thriller"
354,355,High Life,2018,waiting,movie,US,"sci-fi,thriller"
355,356,Saltburn,2023,waiting,movie,US,"thriller,dark comedy,psychological,drama"
356,357,Bloody Flower,2026,waiting,series,Korea,"mystery,crime,thriller,psychological"
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi"


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(features)

In [22]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=df['id'],
    columns=df['id']
)
similarity_df.tail()

id,1,2,3,4,5,6,7,8,9,10,...,349,350,351,352,353,354,355,356,357,358
id,,,,,,,,,,,,,,,,,,,,,
354,0.296082,0.323801,0.181572,0.365892,0.179276,0.154440,0.183849,0.167589,0.201161,0.205995,...,0.439224,0.389562,0.159049,0.377314,0.173960,1.000000,0.792853,0.671043,0.332022,0.543290
355,0.293159,0.319193,0.172274,0.360260,0.170096,0.146532,0.174434,0.159007,0.190860,0.195447,...,0.430804,0.382272,0.150904,0.370897,0.351750,0.792853,1.000000,0.668787,0.325657,0.725146
356,0.250204,0.273922,0.154998,0.309617,0.153038,0.131837,0.156941,0.143061,0.171720,0.175846,...,0.372015,0.500963,0.135771,0.493781,0.148500,0.671043,0.668787,1.000000,0.424950,0.458999
357,0.252511,0.277318,0.332022,0.137730,0.330462,0.294527,0.333562,0.304061,0.370802,0.564768,...,0.377964,0.165077,0.284229,0.324014,0.308607,0.332022,0.325657,0.424950,1.000000,0.154303
358,0.450284,0.473220,0.173960,0.338855,0.171760,0.318126,0.176141,0.328423,0.192727,0.197359,...,0.408248,0.361930,0.461624,0.349975,0.500000,0.543290,0.725146,0.458999,0.154303,1.000000


In [23]:
similarity_matrix[-5:][-5:]

array([[0.29608188, 0.3238014 , 0.18157205, ..., 0.6710426 , 0.33202204,
        0.5432898 ],
       [0.29315913, 0.31919322, 0.1722743 , ..., 0.6687868 , 0.32565707,
        0.72514581],
       [0.25020397, 0.27392184, 0.15499773, ..., 1.        , 0.4249502 ,
        0.45899868],
       [0.25251125, 0.27731847, 0.33202204, ..., 0.4249502 , 1.        ,
        0.15430335],
       [0.45028379, 0.47321998, 0.17395979, ..., 0.45899868, 0.15430335,
        1.        ]])

In [24]:
similarity_df.shape

(358, 358)

## Movies recommendation

In [25]:
def recommend(movie_id, top_k=5):
    
    if movie_id not in similarity_df.index:
        return f"Movie ID {movie_id} not found."

    sim_scores = similarity_df.loc[movie_id].drop(movie_id)  # remove itself

    # Remove completed and dropped movies
    valid_ids = df[df['status'] == 'waiting']['id']
    sim_scores = sim_scores[sim_scores.index.isin(valid_ids)]
    
    top_movies = sim_scores.sort_values(ascending=False).head(top_k)    

    return df[df['id'].isin(top_movies.index)]

In [26]:
import sqlite3
from utils.movie import get_movie

con.row_factory = sqlite3.Row  # for dictionary conversion
cur = con.cursor()

In [27]:
def recsys(movie_id):
    movie = dict(get_movie(movie_id, cur))
    movie.pop('note')
    print(movie)
    display(recommend(movie_id))

In [28]:
recsys(227)

{'id': 227, 'name': 'Go Back Couple', 'year': 2017, 'status': 'completed', 'type': 'series', 'country': 'Korea', 'genres': 'comedy,romance,time travel,life', 'rating': 10.0, 'watched_date': '2025-10-13'}


,id,name,year,status,type,country,genres
161,162,When Life Gives You Tangerines,2025,waiting,series,Korea,"romance,life"
181,182,Reply 1988,2015,waiting,series,Korea,"comedy,romance,life"
193,194,My Liberation Notes,2022,waiting,series,Korea,"romance,life"
230,231,Because This Is My First Life,2017,waiting,series,Korea,"comedy,romance,life"
326,327,Can This Love Be Translated?,2026,waiting,series,Korea,"comedy,romance"


In [29]:
recsys(300)

{'id': 300, 'name': 'Rango', 'year': 2011, 'status': 'completed', 'type': 'movie', 'country': 'US', 'genres': 'comedy,animation,adventure', 'rating': 8.0, 'watched_date': '2025-11-24'}


,id,name,year,status,type,country,genres
94,95,Flow,2024,waiting,movie,US,"animation,adventure"
105,106,The Intern,2015,waiting,movie,US,comedy
222,223,"The Boy, the Mole, the Fox and the Horse",2022,waiting,movie,US,"animation,family,adventure"
346,347,Ralph Breaks the Internet,2018,waiting,movie,US,"comedy,animation,adventure"
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi"


In [30]:
recsys(340)

{'id': 340, 'name': 'Ne Zha (Na Tra)', 'year': 2019, 'status': 'completed', 'type': 'movie', 'country': 'China', 'genres': 'animation,family,adventure,fantasy', 'rating': 9.0, 'watched_date': '2026-01-25'}


,id,name,year,status,type,country,genres
70,71,Deep Sea,2023,waiting,movie,China,"animation,adventure,fantasy"
154,155,The Legend of Hei,2019,waiting,movie,China,"animation,action,adventure,fantasy"
233,234,The Legend of Hei 2,2025,waiting,movie,China,"animation,action,adventure,fantasy"
304,305,Nobody,2025,waiting,movie,China,"comedy,animation,adventure,fantasy"
340,341,Ne Zha 2 (Na Tra 2),2025,waiting,movie,China,"animation,action,adventure,fantasy"


In [31]:
recsys(248)

{'id': 248, 'name': 'Mickey 17', 'year': 2025, 'status': 'completed', 'type': 'movie', 'country': 'US', 'genres': 'adventure,sci-fi,dark comedy', 'rating': 8.0, 'watched_date': '2025-12-14'}


,id,name,year,status,type,country,genres
55,56,Dune,2021,waiting,movie,US,"action,adventure,sci-fi"
80,81,The Martian,2015,waiting,movie,US,"adventure,sci-fi"
110,111,Companion,2025,waiting,movie,US,"sci-fi,thriller,dark comedy"
115,116,Poor Things,2023,waiting,movie,US,"romance,sci-fi,dark comedy"
342,343,Project Hail Mary,2026,waiting,movie,US,"adventure,sci-fi,mystery"


In [32]:
recsys(32)

{'id': 32, 'name': '18 Again', 'year': 2020, 'status': 'completed', 'type': 'series', 'country': 'Korea', 'genres': 'comedy,romance,time travel,family,fantasy', 'rating': 10.0, 'watched_date': '2024-08-31'}


,id,name,year,status,type,country,genres
190,191,Mr. Plankton,2024,waiting,series,Korea,"comedy,romance,death"
194,195,Goblin,2016,waiting,series,Korea,"comedy,romance,fantasy"
211,212,When the Camellia Blooms,2019,waiting,series,Korea,"comedy,romance,family,thriller"
318,319,Spirit Fingers,2025,waiting,series,Korea,"comedy,romance,youth"
326,327,Can This Love Be Translated?,2026,waiting,series,Korea,"comedy,romance"


## Hybrid Recommender Sytem

- [ ] Dimension reduction if it slow (SVD, PCA)

In [96]:
# # Split by status
# # completed + high rating	strong like	strong positive
# # completed + low rating	watched but disliked	weak negative
# # dropped (+ any rating)	rejection	strong negative
# completed = df[df["status"] == "completed"]
# dropped   = df[df["status"] == "dropped"]

# min_r, max_r = 3, 10

# pos_weights = completed["rating"].fillna((min_r + max_r) / 2)
# pos_weights = (pos_weights - min_r) / (max_r - min_r)
# pos_weights = pos_weights.clip(lower=0.3)

# positive_profile = (
#     item_features.loc[completed.index]
#     .mul(pos_weights.values, axis=0)
#     .mean()
# )

# # substract negative profile
# neg_weights = 1.0  # or tune (0.5–1.5)

# negative_profile = (
#     item_features.loc[dropped.index]
#     .mean()
# )

# user_vector = positive_profile - neg_weights * negative_profile

# scores = item_features.dot(user_vector)

In [ ]:
# time decay
# decay = np.exp(-lambda_ * days_since_watch)

## Clean up

In [ ]:
con.close()